In [0]:
SELECT COUNT(*) as total FROM proyecto.bronze.properties_bronze;

total
1246717


In [0]:
DESCRIBE proyecto.bronze.properties_bronze

col_name,data_type,comment
id,double,null
ubicacion,string,null
precio,string,null
numero,string,null
calle,string,null
expensas,string,null
tipo_de_operacion,string,null
moneda,string,null
ambientes,string,null
metros_cuadrados_totales,string,null


In [0]:
-- ====================================================================
-- EDA: Análisis Integral de Nulos y Densidad de Datos
-- ====================================================================
WITH null_metrics AS (
    SELECT
        COUNT(*) AS total_records,
        -- Conteo de nulos
        COUNT(*) - COUNT(id) AS nulos_id,
        COUNT(*) - COUNT(ubicacion) AS nulos_ubicacion,
        COUNT(*) - COUNT(precio) AS nulos_precio,
        COUNT(*) - COUNT(numero) AS nulos_numero,
        COUNT(*) - COUNT(calle) AS nulos_calle,
        COUNT(*) - COUNT(expensas) AS nulos_expensas,
        COUNT(*) - COUNT(tipo_de_operacion) AS nulos_tipo_operacion,
        COUNT(*) - COUNT(moneda) AS nulos_moneda,
        COUNT(*) - COUNT(ambientes) AS nulos_ambientes,
        COUNT(*) - COUNT(metros_cuadrados_totales) AS nulos_m2_totales,
        COUNT(*) - COUNT(metros_cuadrados_cubiertos) AS nulos_m2_cubiertos,
        COUNT(*) - COUNT(orientacion_cardinal) AS nulos_orientacion_cardinal,
        COUNT(*) - COUNT(orientacion_inmueble) AS nulos_orientacion_inmueble,
        COUNT(*) - COUNT(piso) AS nulos_piso,
        COUNT(*) - COUNT(cochera) AS nulos_cochera,
        COUNT(*) - COUNT(antiguedad) AS nulos_antiguedad,
        COUNT(*) - COUNT(estado) AS nulos_estado,
        COUNT(*) - COUNT(tipo_vendedor) AS nulos_tipo_vendedor,
        COUNT(*) - COUNT(url) AS nulos_url,
        COUNT(*) - COUNT(zona) AS nulos_zona,
        COUNT(*) - COUNT(fecha) AS nulos_fecha,
        COUNT(*) - COUNT(hora) AS nulos_hora,
        COUNT(*) - COUNT(_rescued_data) AS nulos_rescued_data,
        COUNT(*) - COUNT(ingestion_timestamp) AS nulos_ingestion_timestamp,
        COUNT(*) - COUNT(source_file) AS nulos_source_file
    FROM proyecto.bronze.properties_bronze
),
unpivoted_nulls AS (
    SELECT 
        REPLACE(col_name, 'nulos_', '') AS column_name,
        null_count,
        ROUND((null_count * 100.0) / total_records, 2) AS pct_nulls
    FROM null_metrics
    UNPIVOT (
        null_count FOR col_name IN (
            nulos_id, nulos_ubicacion, nulos_precio, nulos_numero, nulos_calle, nulos_expensas,
            nulos_tipo_operacion, nulos_moneda, nulos_ambientes, nulos_m2_totales, nulos_m2_cubiertos, nulos_orientacion_cardinal, nulos_orientacion_inmueble, nulos_piso, nulos_cochera, nulos_antiguedad, nulos_estado, nulos_tipo_vendedor, nulos_url, nulos_zona, nulos_fecha, nulos_hora, nulos_rescued_data, nulos_ingestion_timestamp, nulos_source_file
        )
    )
)
SELECT 
    column_name,
    null_count,
    pct_nulls,
    CASE 
        WHEN pct_nulls > 50 THEN 'Descartar o Imputar en Silver (>50% null)'
        WHEN pct_nulls > 0 THEN 'Imputar o Filtrar'
        ELSE 'Limpia / Clave'
    END AS action_flag
FROM unpivoted_nulls
ORDER BY pct_nulls DESC;

column_name,null_count,pct_nulls,action_flag
rescued_data,1246717,100.00,Descartar o Imputar en Silver (>50% null)
orientacion_cardinal,1152691,92.46,Descartar o Imputar en Silver (>50% null)
tipo_vendedor,1054435,84.58,Descartar o Imputar en Silver (>50% null)
orientacion_inmueble,1042174,83.59,Descartar o Imputar en Silver (>50% null)
piso,959140,76.93,Descartar o Imputar en Silver (>50% null)
cochera,744399,59.71,Descartar o Imputar en Silver (>50% null)
estado,737198,59.13,Descartar o Imputar en Silver (>50% null)
expensas,680210,54.56,Descartar o Imputar en Silver (>50% null)
m2_cubiertos,318632,25.56,Imputar o Filtrar
numero,242473,19.45,Imputar o Filtrar


In [0]:
-- Distribution of 'tipo de operacion' (operation type)

SELECT 
    tipo_de_operacion,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 4) AS pct
FROM proyecto.bronze.properties_bronze
GROUP BY  tipo_de_operacion
ORDER BY total DESC;

tipo_de_operacion,total,pct
alquiler,626540,50.2552
venta,603346,48.3948
alquiler_temporal,14524,1.1650
null,1458,0.1169
alquiler_de_habitacion,206,0.0165
Las Malvinas,49,0.0039
alquiler_anual,35,0.0028
venta/alquiler,19,0.0015
Emilio Lamarca,18,0.0014
las malvinas,16,0.0013


In [0]:
-- Distribution of 'moneda' (type of currency)

SELECT
    moneda,
    COUNT(*) AS total_moneda,
    ROUND(COUNT(*) * 100.00 / SUM(COUNT(*)) OVER(), 4) AS pct
FROM proyecto_prueba.bronze.properties_bronze
GROUP BY moneda
ORDER BY total_moneda DESC

moneda,total_moneda,pct
USD,851579,68.3378
ARS,388893,31.2080
null,5643,0.4528
MXN,8,0.0006
consultar,2,0.0002
guaranies,2,0.0002
uyu,2,0.0002
null,1,0.0001
ar,1,0.0001


In [0]:
-- Ambients Distribution

SELECT
    CAST(ambientes AS FLOAT) AS ambientes,
    COUNT(*) AS total_ambiente,
    ROUND(COUNT(*) * 100.00 / SUM(count(*)) OVER(), 2) AS PCT_ambiente 
FROM proyecto_prueba.bronze.properties_bronze
GROUP BY ambientes
ORDER BY total_ambiente DESC

ambientes,total_ambiente,PCT_ambiente
2.0,343223,27.54
3.0,326635,26.21
4.0,218585,17.54
1.0,156331,12.55
5.0,104737,8.40
6.0,46063,3.70
7.0,20143,1.62
null,13566,1.09
8.0,7669,0.62
10.0,2655,0.21


In [0]:
-- TOP zonas (top areas with most listed properties)
WITH tabla1 AS(
    SELECT
        zona,
        COUNT(*) AS total_zona,
        ROUND(COUNT(*) * 100.00 / SUM(count(*)) OVER(), 2) AS pct_zona 
    FROM proyecto_prueba.bronze.properties_bronze
    GROUP BY zona
    ORDER BY total_zona DESC
)

SELECT 
    *
FROM tabla1
WHERE pct_zona >= 0.5

zona,total_zona,pct_zona
capital-federal,450364,36.14
tigre,91470,7.34
pilar,80214,6.44
vicente-lopez,67555,5.42
escobar,57302,4.60
san-isidro,47597,3.82
general-san-martin,42987,3.45
san-miguel,30786,2.47
san-fernando,30643,2.46
jose-c-paz,16576,1.33


In [0]:
-- Status Distribution

WITH tabla1 AS (
    SELECT
        estado,
        COUNT(*) AS total_estado,
        ROUND(COUNT(*) * 100.00 / SUM(count(*)) OVER(), 2) AS pct_estado 
    FROM proyecto_prueba.bronze.properties_bronze
    GROUP BY estado
    ORDER BY total_estado DESC
)

SELECT 
    *
FROM tabla1

estado,total_estado,pct_estado
null,736825,59.13
excelente,370092,29.70
bueno,109399,8.78
a_refaccionar,26521,2.13
muy bueno,1076,0.09
a_estrenar,849,0.07
a estrenar,362,0.03
impecable,297,0.02
0,191,0.02
terminada,124,0.01


In [0]:
-- Temporary View to Expand the EDA (CAST all STRING number values to FLOAT)

CREATE OR REPLACE TEMPORARY VIEW properties_clean AS
SELECT 
    id,
    CASE
        WHEN precio RLIKE '^[0-9]+(\.[0-9]+)?$' THEN precio::double
        ELSE NULL
    END as precio,
    moneda,
    CASE
     WHEN ambientes RLIKE '^[0-9]+(\.[0-9]+)?$' THEN ambientes::double
        ELSE NULL
    END as ambientes
    ,
    CASE
        WHEN metros_cuadrados_totales RLIKE '^[0-9]+(\.[0-9]+)?$' THEN metros_cuadrados_totales::double
        ELSE NULL
    END as m2_totales
    ,
    CASE
        WHEN metros_cuadrados_cubiertos RLIKE '^[0-9]+(\.[0-9]+)?$' THEN metros_cuadrados_cubiertos::double
        ELSE NULL
    END as m2_cubiertos
    ,
    CASE
        WHEN antiguedad RLIKE '^[0-9]+(\.[0-9]+)?$' THEN antiguedad::double
        ELSE NULL
    END as antiguedad,
    tipo_de_operacion,
    ubicacion,
    expensas,
    piso,
    cochera,
    estado,
    url,
    zona,
    fecha
FROM proyecto.bronze.properties_bronze;

SELECT COUNT(*) FROM properties_clean;

COUNT(*)
1246717


In [0]:
-- Currency and Operation type Analysis

SELECT 
    moneda,
    tipo_de_operacion,
    COUNT(*) as cantidad,
    ROUND(MIN(precio), 2) as precio_min,
    ROUND(MAX(precio), 2) as precio_max,
    ROUND(AVG(precio), 2) as precio_promedio,
    ROUND(PERCENTILE(precio, 0.05), 2) as precio_p05,
    ROUND(PERCENTILE(precio, 0.10), 2) as precio_p10,
    ROUND(PERCENTILE(precio, 0.25), 2) as precio_p25,
    ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,
    ROUND(PERCENTILE(precio, 0.75), 2) as precio_p75,
    ROUND(PERCENTILE(precio, 0.90), 2) as precio_p90,
    ROUND(PERCENTILE(precio, 0.95), 2) as precio_p95
FROM properties_clean -- Recent View
WHERE precio IS NOT NULL 
    AND precio > 0
    AND tipo_de_operacion IN ('venta', 'alquiler')
    AND moneda IN ('USD', 'ARS')
GROUP BY moneda, tipo_de_operacion
ORDER BY moneda, tipo_de_operacion;

moneda,tipo_de_operacion,cantidad,precio_min,precio_max,precio_promedio,precio_p05,precio_p10,precio_p25,precio_mediana,precio_p75,precio_p90,precio_p95
ARS,alquiler,374379,1.0,1.2E9,793701.4,350000.0,400000.0,500000.0,650000.0,900000.0,1300000.0,1690000.0
ARS,venta,9566,1.0,1.434768228E9,9169126.52,59000.0,90000.0,240000.0,700000.0,1400000.0,2600000.0,3105787.5
USD,alquiler,248213,1.0,2.14423E7,18585.05,550.0,650.0,1000.0,1600.0,3000.0,5000.0,8000.0
USD,venta,589016,0.06,1.111111111E9,219097.52,38000.0,51000.0,79500.0,135000.0,241200.0,420000.0,620000.0


In [0]:
CREATE OR REPLACE TEMPORARY VIEW properties_USD AS (
  SELECT
    id, 
    tipo_de_operacion,
    CASE 
        WHEN moneda = 'ARS' THEN ROUND(precio / 1520, -1)
        ELSE precio
    END AS precio,
    ambientes,
    m2_totales,
    m2_cubiertos,
    antiguedad,
    ubicacion,
    expensas,
    piso,
    cochera,
    estado,
    url,
    zona,
    fecha
FROM properties_clean
WHERE 
  precio IS NOT NULL
  AND precio > 0
  AND tipo_de_operacion IN ('venta', 'alquiler')
  AND moneda IN ('ARS','USD')
);


SELECT COUNT(*) FROM properties_USD;

count(*)
1221174


In [0]:
    -- deteccion de outliers en el precio
    
    SELECT 
      tipo_de_operacion,
      PERCENTILE(precio, 0.05) AS p05,
      PERCENTILE(precio, 0.95) AS p95
    FROM properties_USD
    GROUP BY tipo_de_operacion

tipo_de_operacion,p05,p95
alquiler,250.0,4500.0
venta,32000.0,620000.0


In [0]:
-- Vista nueva todo en USD y sin los outliers en el precio

CREATE OR REPLACE TEMPORARY VIEW properties_clean_2 AS
(
WITH limites AS (
    SELECT 
      tipo_de_operacion,
      PERCENTILE(precio, 0.05) AS p05,
      PERCENTILE(precio, 0.95) AS p95
    FROM properties_USD
    GROUP BY tipo_de_operacion
  )
  SELECT p.*
  FROM properties_usd AS p
  JOIN limites AS l 
    ON p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p05 AND l.p95
);

SELECT COUNT(*) FROM properties_clean_2;

COUNT(*)
1103035


In [0]:
-- Metros cuadrados totales vs metros cuadrados cubiertos
-- Deteccion de Outliers

SELECT 
    'metros_cuadrados_totales' AS columna,
    COUNT(*) AS cantidad_no_nulos,
    ROUND(MIN(m2_totales), 2) AS minimo,
    ROUND(MAX(m2_totales), 2) AS maximo,
    ROUND(AVG(m2_totales), 2) AS promedio,
    ROUND(PERCENTILE(m2_totales, 0.001), 2) AS p001,
    ROUND(PERCENTILE(m2_totales, 0.5), 2) AS mediana,
    ROUND(PERCENTILE(m2_totales, 0.999), 2) AS p999
FROM 
    properties_clean_2
WHERE 
    m2_totales IS NOT NULL 
    AND m2_totales > 0

UNION ALL

SELECT 
    'metros_cuadrados_cubiertos' AS columna,
    COUNT(*) AS cantidad_no_nulos,
    ROUND(MIN(m2_cubiertos), 2) AS minimo,
    ROUND(MAX(m2_cubiertos), 2) AS maximo,
    ROUND(AVG(m2_cubiertos), 2) AS promedio,
    ROUND(PERCENTILE(m2_cubiertos, 0.001), 2) AS p001,
    ROUND(PERCENTILE(m2_cubiertos, 0.5), 2) AS mediana,
    ROUND(PERCENTILE(m2_cubiertos, 0.999), 2) AS p999
FROM 
    properties_clean_2
WHERE 
    m2_cubiertos IS NOT NULL 
    AND m2_cubiertos > 0;

columna,cantidad_no_nulos,minimo,maximo,promedio,p001,mediana,p999
metros_cuadrados_totales,1059296,1.0,2.147483647E9,9536.66,18.0,76.0,8428.0
metros_cuadrados_cubiertos,815280,1.0,2.0E9,7473.9,17.0,72.0,600.0


In [0]:
-- Deteccion de Duplicaods

WITH tabla1 AS(
	SELECT
		COUNT(*) AS repetidos,
		precio,
		url,
		COUNT(*) - 1 AS registros_extras
	FROM properties_clean_2
	GROUP BY precio, url
	HAVING COUNT(*) > 1
	ORDER BY repetidos DESC
)
SELECT
	COUNT(*) AS registros_repetidos,
	SUM(repetidos) AS total_registros_repetidos,
	SUM(repetidos) - COUNT(*) AS total_extras
FROM tabla1

registros_repetidos,total_registros_repetidos,total_extras
71791,184041,112250


In [0]:
-- Ejemplos de Duplicaods

SELECT
    COUNT(*) as repetidos,
    precio,
    url
FROM properties_clean_2
GROUP BY  precio, url
HAVING COUNT(*) > 1
ORDER BY repetidos DESC
LIMIT 5

repetidos,precio,url
12,62000.0,https://departamento.mercadolibre.com.ar/MLA-2059467746-venta-departamento-2-ambientes-con-cochera-a-estrenar-en-villa-luzuriaga-uf-4-_JM
11,56000.0,https://departamento.mercadolibre.com.ar/MLA-1507670893-venta-departamento-3-ambientes-lateral-en-ramos-mejia-_JM
10,1640.0,https://www.argenprop.com/casa-en-alquiler-en-san-miguel-5-ambientes--17612817
10,55000.0,https://www.argenprop.com/departamento-en-venta-en-moreno-2-ambientes--17694403
10,69000.0,https://departamento.mercadolibre.com.ar/MLA-1487887695-venta-departamento-2-ambientes-en-lomas-del-mirador-2a-_JM
